# Top 6 Station H5 Model Store

상위 6개 station 모델을 하나의 `h5` 파일로 통합 저장하고, 실제 서비스에서 로드한 뒤 입력값으로 예측하고 JSON으로 출력하는 전체 과정을 정리한 노트북입니다.

In [1]:
from pathlib import Path
import json
import h5py
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[0]
DATA_DIR = ROOT / 'Data'
NOTE_DIR = ROOT / 'Note'
H5_PATH = DATA_DIR / 'top6_station_model_store.h5'
TOP6_STATION_IDS = [2348, 2335, 2377, 2384, 2306, 2375]


## 0. 서비스 입력값과 출력값

이 모델은 외부에서 많은 feature를 직접 받지 않습니다. 서비스에서 **직접 넣어야 하는 입력값은 아래 3개**입니다.

- `station_id`: 어떤 대여소 모델을 쓸지 결정
- `datetime`: 예측하려는 시점
- `current_bike_count_from_api`: 현재 자전거 수량 API 값

이 중 `station_id`와 `datetime`은 단일 시점 예측에 필요하고, `current_bike_count_from_api`는 재귀형 다중 시간 예측에 필요합니다.

최종 출력은 JSON입니다. JSON 안에는 보통 아래 값이 들어갑니다.

- `rental_count_pred`
- `return_count_pred`
- `bike_change_pred`
- `next_bike_count_pred` 또는 이후 시간대 예측 배열

In [2]:
service_io_df = pd.DataFrame([
    {'field': 'station_id', 'required': True, 'type': 'int', 'description': '예측에 사용할 station 모델 id'},
    {'field': 'datetime', 'required': True, 'type': 'datetime or ISO string', 'description': '예측 대상 시점'},
    {'field': 'current_bike_count_from_api', 'required': False, 'type': 'float', 'description': '현재 자전거 수량. 재귀형 예측 시작값'},
])
service_io_df

,field,required,type,description
0,station_id,True,int,예측에 사용할 station 모델 id
1,datetime,True,datetime or ISO string,예측 대상 시점
2,current_bike_count_from_api,False,float,현재 자전거 수량. 재귀형 예측 시작값


### 입력 항목 상세 설명

- `station_id`
  - 어떤 대여소 모델을 사용할지 결정하는 식별자입니다.
  - 이 값으로 `MODEL_REGISTRY`에서 station별 공식, 가중치, Ridge 계수를 선택합니다.
- `datetime`
  - 예측하려는 기준 시각입니다.
  - 이 값에서 `year`, `month`, `day`, `hour`, `weekday`를 추출합니다.
  - 공휴일 여부와 함께 `day_type`를 판별하는 핵심 입력입니다.
- `current_bike_count_from_api`
  - 현재 시점 실제 자전거 총 대수입니다.
  - 단일 시점 예측 자체에는 직접 들어가지 않지만, 재귀형 예측에서는 다음 시간 총 대수를 계산하는 시작값이 됩니다.

## 1. 상위 6개 station 확인

통합 저장 대상은 `test R²` 기준 상위 6개 station입니다.

In [3]:
ranking_df = pd.read_csv(DATA_DIR / 'summaries/top20_station_combined_test_r2_ranking.csv')
top6_df = ranking_df.head(6).copy()
top6_df[['rank', 'station_id', 'station_name', 'combined_test_r2', 'combined_test_rmse', 'combined_test_mae']]

,rank,station_id,station_name,combined_test_r2,combined_test_rmse,combined_test_mae
0,1,2348,포스코사거리(기업은행),0.614441,1.971680,1.150548
1,2,2335,3호선 매봉역 3번출구앞,0.545963,1.806116,1.222398
2,3,2377,수서역 5번출구,0.455599,1.934291,1.326159
3,4,2384,자곡사거리,0.435526,1.526692,1.065443
4,5,2306,압구정역 2번 출구 옆,0.431629,1.473790,1.002118
5,6,2375,수서역 1번출구 앞,0.408915,1.353031,0.941724


## 2. station별 bundle 만들기

각 station에서 아래 정보를 모아 하나의 bundle로 만듭니다.

- Ridge 계수와 intercept
- feature_names
- hour 기본식
- month/year/hour weight

In [4]:
def build_station_bundle(station_id):
    coef_df = pd.read_csv(DATA_DIR / 'coefficients' / f'station_{station_id}_offday_month_ridge_coefficients.csv')
    formula_df = pd.read_csv(DATA_DIR / 'formulas' / f'station_{station_id}_offday_hour_formulas.csv')
    weight_df = pd.read_csv(DATA_DIR / 'weights' / f'station_{station_id}_month_weights.csv')

    bundle = {
        'station_id': int(station_id),
        'formula': {},
        'month_weight': {},
        'year_weight': {},
        'hour_weight': {},
    }

    for target in ['rental_count', 'return_count']:
        target_coef = coef_df[coef_df['target'] == target].copy()
        feature_df = target_coef[target_coef['feature'] != 'intercept'].copy()
        intercept = float(target_coef.loc[target_coef['feature'] == 'intercept', 'coefficient'].iloc[0])
        bundle_key = 'rental' if target == 'rental_count' else 'return'
        bundle[bundle_key] = {
            'coef': feature_df['coefficient'].astype(float).tolist(),
            'intercept': intercept,
            'feature_names': feature_df['feature'].tolist(),
        }

        target_formula = formula_df[formula_df['target'] == target].copy()
        bundle['formula'][target] = {}
        for _, row in target_formula.iterrows():
            bundle['formula'][target][row['day_type']] = {
                'intercept': float(row['intercept']),
                'sin_hour_coef': float(row['sin_hour_coef']),
                'cos_hour_coef': float(row['cos_hour_coef']),
            }

        target_weights = weight_df[weight_df['target'] == target].copy()
        for weight_type in ['month_weight', 'year_weight', 'hour_weight']:
            sub = target_weights[target_weights['weight_type'] == weight_type].copy()
            bundle[weight_type][target] = {str(row['key']): float(row['value']) for _, row in sub.iterrows()}

    return bundle


In [5]:
example_bundle = build_station_bundle(TOP6_STATION_IDS[0])
example_bundle.keys()

dict_keys(['station_id', 'formula', 'month_weight', 'year_weight', 'hour_weight', 'rental', 'return'])

## 3. 통합 h5 저장

모든 station을 하나의 `top6_station_model_store.h5` 파일에 저장합니다.

In [6]:
holiday_df = pd.read_csv(DATA_DIR / 'holiday_reference' / f'station_{TOP6_STATION_IDS[0]}_holiday_reference.csv')
holiday_dates = sorted(pd.to_datetime(holiday_df['date']).dt.strftime('%Y-%m-%d').unique().tolist())

def save_model_store(h5_path, station_ids, holiday_dates):
    if h5_path.exists():
        h5_path.unlink()
    with h5py.File(h5_path, 'w') as f:
        meta_grp = f.require_group('meta')
        meta_grp.attrs['model_type'] = 'pattern_weight_ridge'
        meta_grp.attrs['version'] = 'v1'
        meta_grp.create_dataset('station_ids', data=np.asarray(station_ids, dtype=int))
        meta_grp.create_dataset('holiday_dates', data=np.asarray(holiday_dates, dtype='S'))

        stations_grp = f.require_group('stations')
        ranking_df = pd.read_csv(DATA_DIR / 'summaries/top20_station_combined_test_r2_ranking.csv')
        for station_id in station_ids:
            bundle = build_station_bundle(station_id)
            station_name = ranking_df.loc[ranking_df['station_id'] == station_id, 'station_name'].iloc[0]
            st_grp = stations_grp.require_group(str(station_id))
            st_grp.attrs['station_name'] = station_name

            for target_key in ['rental', 'return']:
                target_grp = st_grp.require_group(target_key)
                target_bundle = bundle[target_key]
                target_grp.create_dataset('coef', data=np.asarray(target_bundle['coef'], dtype=float))
                target_grp.create_dataset('intercept', data=np.asarray([target_bundle['intercept']], dtype=float))
                target_grp.create_dataset('feature_names', data=np.asarray(target_bundle['feature_names'], dtype='S'))

            formula_grp = st_grp.require_group('formula')
            for target in ['rental_count', 'return_count']:
                target_formula_grp = formula_grp.require_group(target)
                for day_type, params in bundle['formula'][target].items():
                    day_grp = target_formula_grp.require_group(day_type)
                    for key, value in params.items():
                        day_grp.attrs[key] = float(value)

            for weight_type in ['month_weight', 'year_weight', 'hour_weight']:
                outer_grp = st_grp.require_group(weight_type)
                for target in ['rental_count', 'return_count']:
                    target_grp = outer_grp.require_group(target)
                    keys = np.asarray(list(bundle[weight_type][target].keys()), dtype='S')
                    values = np.asarray(list(bundle[weight_type][target].values()), dtype=float)
                    target_grp.create_dataset('keys', data=keys)
                    target_grp.create_dataset('values', data=values)

save_model_store(H5_PATH, TOP6_STATION_IDS, holiday_dates)
H5_PATH

WindowsPath('C:/Users/TJ/ddri_work/hmw3/Data/top6_station_model_store.h5')

## 4. 서비스용 로더

실서비스에서는 서버 시작 시 이 함수를 한 번 실행해서 station 모델을 메모리에 올려두는 방식이 가장 효율적입니다.

In [7]:
def load_model_store(h5_path):
    registry = {}
    with h5py.File(h5_path, 'r') as f:
        holiday_dates = {x.decode('utf-8') for x in f['meta']['holiday_dates'][:]}
        station_ids = [int(x) for x in f['meta']['station_ids'][:].tolist()]
        for station_id in station_ids:
            st_grp = f['stations'][str(station_id)]
            station_bundle = {
                'station_id': station_id,
                'station_name': st_grp.attrs['station_name'],
                'formula': {},
                'month_weight': {},
                'year_weight': {},
                'hour_weight': {},
            }
            for target_key in ['rental', 'return']:
                t_grp = st_grp[target_key]
                station_bundle[target_key] = {
                    'coef': t_grp['coef'][:],
                    'intercept': float(t_grp['intercept'][0]),
                    'feature_names': [x.decode('utf-8') for x in t_grp['feature_names'][:]],
                }
            for target in ['rental_count', 'return_count']:
                station_bundle['formula'][target] = {}
                for day_type in ['weekday', 'offday']:
                    attrs = st_grp['formula'][target][day_type].attrs
                    station_bundle['formula'][target][day_type] = {key: float(attrs[key]) for key in attrs.keys()}
            for weight_type in ['month_weight', 'year_weight', 'hour_weight']:
                station_bundle[weight_type] = {}
                for target in ['rental_count', 'return_count']:
                    grp = st_grp[weight_type][target]
                    keys = [x.decode('utf-8') for x in grp['keys'][:]]
                    values = grp['values'][:].tolist()
                    station_bundle[weight_type][target] = dict(zip(keys, values))
            registry[station_id] = station_bundle
    return registry, holiday_dates

MODEL_REGISTRY, HOLIDAY_DATES = load_model_store(H5_PATH)
list(MODEL_REGISTRY.keys())

[2348, 2335, 2377, 2384, 2306, 2375]

## 5. 입력값으로 feature 만들기

서비스에서는 `station_id`와 `datetime`을 받으면 같은 로직으로 feature를 구성해야 합니다.

### 입력값이 내부 feature로 바뀌는 과정

외부 입력값은 단순하지만, 모델이 실제로 사용하는 값은 아래 8개 내부 feature입니다.

- `base_value`
- `month_weight`
- `year_weight`
- `hour_weight`
- `pattern_prior`
- `corrected_pattern_prior`
- `day_type_weekday`
- `day_type_offday`

변환 순서는 다음과 같습니다.

1. `datetime`에서 `year`, `month`, `day`, `hour`, `weekday`를 계산
2. 공휴일 테이블을 보고 `day_type = weekday / offday` 결정
3. `station_id`에 해당하는 기본식으로 `base_value` 계산
4. 같은 station의 `month_weight`, `year_weight`, `hour_weight` 조회
5. `pattern_prior = base_value * month_weight * year_weight`
6. `corrected_pattern_prior = pattern_prior * hour_weight`
7. `day_type_weekday`, `day_type_offday` 더미값 생성
8. 이 8개 값을 Ridge 입력 feature로 사용

### 내부 feature 상세 설명

- `base_value`
  - `day_type`와 `hour`를 이용해 기본 시간 패턴식에서 계산한 값입니다.
  - station이 원래 가지는 시간대별 평균적인 흐름을 나타냅니다.
- `month_weight`
  - 월별 규모 차이를 보정하는 가중치입니다.
  - 예를 들어 5월 이용량이 1월보다 전반적으로 크면 5월 weight가 더 크게 설정됩니다.
- `year_weight`
  - 2023~2025 학습 데이터에서 관측된 연도별 규모 차이를 반영하는 가중치입니다.
  - 2026 이후에는 이 고정값 위에 `dynamic_year_weight`를 추가해 운영 중 보정합니다.
- `hour_weight`
  - 기본 시간 패턴식으로 설명되지 않는 피크 시간대 편차를 보정하는 가중치입니다.
  - 출퇴근 시간처럼 특정 시간대가 더 튀는 경우를 보완합니다.
- `pattern_prior`
  - `base_value * month_weight * year_weight`로 계산되는 1차 패턴 예측값입니다.
  - 시간 패턴과 월/연도 규모를 먼저 결합한 값입니다.
- `corrected_pattern_prior`
  - `pattern_prior * hour_weight`로 계산되는 2차 보정 패턴값입니다.
  - 실제 Ridge 입력에서 가장 중요한 설명 변수로 사용됩니다.
- `day_type_weekday`
  - 평일이면 1, 아니면 0인 더미 변수입니다.
  - 평일 패턴을 별도로 구분하기 위한 feature입니다.
- `day_type_offday`
  - 주말 또는 공휴일이면 1, 아니면 0인 더미 변수입니다.
  - 비근무일 패턴을 별도로 반영하기 위한 feature입니다.

In [8]:
def resolve_weight(weight_map, key):
    if str(key) in weight_map:
        return float(weight_map[str(key)])
    numeric_keys = sorted(int(k) for k in weight_map.keys())
    fallback = max(numeric_keys)
    return float(weight_map[str(fallback)])

def compute_base_value(formula_params, hour):
    angle = 2 * np.pi * hour / 24.0
    return (
        formula_params['intercept']
        + formula_params['sin_hour_coef'] * np.sin(angle)
        + formula_params['cos_hour_coef'] * np.cos(angle)
    )

def build_features(bundle, dt, target):
    date_str = dt.strftime('%Y-%m-%d')
    day_type = 'offday' if (dt.weekday() >= 5 or date_str in HOLIDAY_DATES) else 'weekday'
    base_value = compute_base_value(bundle['formula'][target][day_type], int(dt.hour))
    month_weight = resolve_weight(bundle['month_weight'][target], int(dt.month))
    year_weight = resolve_weight(bundle['year_weight'][target], int(dt.year))
    hour_weight = resolve_weight(bundle['hour_weight'][target], int(dt.hour))
    pattern_prior = base_value * month_weight * year_weight
    corrected_pattern_prior = pattern_prior * hour_weight
    return {
        'base_value': float(base_value),
        'month_weight': float(month_weight),
        'year_weight': float(year_weight),
        'hour_weight': float(hour_weight),
        'pattern_prior': float(pattern_prior),
        'corrected_pattern_prior': float(corrected_pattern_prior),
        'day_type_weekday': 1.0 if day_type == 'weekday' else 0.0,
        'day_type_offday': 1.0 if day_type == 'offday' else 0.0,
    }


In [9]:
feature_flow_example_dt = pd.Timestamp('2026-04-25 16:00:00')
feature_flow_station_id = TOP6_STATION_IDS[0]
feature_flow_bundle = MODEL_REGISTRY[int(feature_flow_station_id)]
feature_flow_rental = build_features(feature_flow_bundle, feature_flow_example_dt, 'rental_count')
pd.DataFrame([
    {'step': 1, 'item': 'station_id', 'value': int(feature_flow_station_id)},
    {'step': 2, 'item': 'datetime', 'value': feature_flow_example_dt.isoformat()},
    {'step': 3, 'item': 'internal_features', 'value': json.dumps(feature_flow_rental, ensure_ascii=False)},
])

,step,item,value
0,1,station_id,2348
1,2,datetime,2026-04-25T16:00:00
2,3,internal_features,"{""base_value"": 1.062892290182576, ""month_weigh..."


## 6. 예측 함수 만들기

최종적으로는 `rental_count`, `return_count`, `bike_change`를 JSON 형태로 반환합니다.

### 예측 결과 항목 설명

- `rental_count_pred`
  - 해당 시간에 예상되는 대여 건수입니다.
  - 내부 계산은 float로 유지됩니다.
- `return_count_pred`
  - 해당 시간에 예상되는 반납 건수입니다.
  - 내부 계산은 float로 유지됩니다.
- `bike_change_pred`
  - `rental_count_pred - return_count_pred`로 계산한 순변화량입니다.
  - 양수면 자전거가 줄어드는 방향, 음수면 자전거가 늘어나는 방향으로 해석할 수 있습니다.
- `display`
  - 사용자 화면이나 운영 대시보드에 보여주기 위한 반올림 정수값입니다.
  - 계산용이 아니라 출력용입니다.

In [10]:
def linear_predict(model_info, feature_dict):
    x = np.asarray([feature_dict[name] for name in model_info['feature_names']], dtype=float)
    return float(model_info['intercept'] + np.dot(model_info['coef'], x))

def predict_station_flow(station_id, dt):
    bundle = MODEL_REGISTRY[int(station_id)]
    rental_features = build_features(bundle, dt, 'rental_count')
    return_features = build_features(bundle, dt, 'return_count')

    rental_pred = max(linear_predict(bundle['rental'], rental_features), 0.0)
    return_pred = max(linear_predict(bundle['return'], return_features), 0.0)
    bike_change = float(rental_pred - return_pred)

    result = {
        'station_id': int(station_id),
        'station_name': bundle['station_name'],
        'datetime': dt.isoformat(),
        'input': {
            'year': int(dt.year),
            'month': int(dt.month),
            'day': int(dt.day),
            'hour': int(dt.hour),
        },
        'prediction': {
            'rental_count_pred': float(rental_pred),
            'return_count_pred': float(return_pred),
            'bike_change_pred': bike_change,
        },
        'display': {
            'rental_count_pred': int(round(rental_pred)),
            'return_count_pred': int(round(return_pred)),
            'bike_change_pred': int(round(bike_change)),
        },
    }
    return result


### 서비스 요청 예시

실서비스에서는 보통 아래처럼 요청이 들어온다고 가정할 수 있습니다.

In [11]:
service_request_example = {
    'station_id': int(TOP6_STATION_IDS[0]),
    'datetime': '2026-04-25T16:00:00',
    'current_bike_count_from_api': 12.0,
}
print(json.dumps(service_request_example, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "datetime": "2026-04-25T16:00:00",
  "current_bike_count_from_api": 12.0
}


### 요청값을 모델에 넣는 과정

서비스 코드는 보통 아래 순서로 동작합니다.

1. 요청 JSON에서 `station_id`, `datetime`을 파싱
2. `station_id`로 `MODEL_REGISTRY`에서 station 모델 선택
3. `datetime`을 `Timestamp`로 변환
4. `build_features(...)`로 내부 feature 생성
5. `linear_predict(...)`로 `rental_count`, `return_count` 예측
6. `bike_change = rental - return` 계산
7. JSON 응답으로 반환

In [12]:
request_dt = pd.Timestamp(service_request_example['datetime'])
request_station_id = int(service_request_example['station_id'])
request_bundle = MODEL_REGISTRY[request_station_id]

rental_feature_dict = build_features(request_bundle, request_dt, 'rental_count')
return_feature_dict = build_features(request_bundle, request_dt, 'return_count')

request_flow_df = pd.DataFrame([
    {'stage': 'request', 'name': 'station_id', 'value': request_station_id},
    {'stage': 'request', 'name': 'datetime', 'value': request_dt.isoformat()},
    {'stage': 'feature_build', 'name': 'rental_features', 'value': json.dumps(rental_feature_dict, ensure_ascii=False)},
    {'stage': 'feature_build', 'name': 'return_features', 'value': json.dumps(return_feature_dict, ensure_ascii=False)},
])
request_flow_df

,stage,name,value
0,request,station_id,2348
1,request,datetime,2026-04-25T16:00:00
2,feature_build,rental_features,"{""base_value"": 1.062892290182576, ""month_weigh..."
3,feature_build,return_features,"{""base_value"": 1.047471076552021, ""month_weigh..."


## 7. JSON 결과 출력 예시

예시 입력 시각: `2026-04-25 16:00:00`

출력 형식은 실제 서비스 응답처럼 JSON 문자열로 만듭니다.

In [13]:
example_dt = pd.Timestamp('2026-04-25 16:00:00')
example_result = predict_station_flow(TOP6_STATION_IDS[0], example_dt)
print(json.dumps(example_result, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "station_name": " 포스코사거리(기업은행)",
  "datetime": "2026-04-25T16:00:00",
  "input": {
    "year": 2026,
    "month": 4,
    "day": 25,
    "hour": 16
  },
  "prediction": {
    "rental_count_pred": 0.9800492643950891,
    "return_count_pred": 0.38491116236471035,
    "bike_change_pred": 0.5951381020303788
  },
  "display": {
    "rental_count_pred": 1,
    "return_count_pred": 0,
    "bike_change_pred": 1
  }
}


## 8. 현재 API 자전거 수량을 이용한 재귀형 7일 예측

이제 단일 시점 예측을 반복 적용합니다.

- 현재 시점의 자전거 수량은 API에서 받습니다.
- 다음 1시간의 `rental_count`, `return_count`를 예측합니다.
- `bike_change = rental_count - return_count`를 계산합니다.
- `다음 시간 자전거 수량 = 현재 자전거 수량 + bike_change` 로 갱신합니다.
- 이 과정을 반복해서 최대 7일, 즉 168시간 뒤까지 예측합니다.

### 재귀형 예측 결과 항목 설명

- `bike_count_from_api_or_prev_step`
  - 현재 step 계산의 기준이 되는 자전거 총 대수입니다.
  - 첫 step에서는 API 실제값이고, 이후 step부터는 직전 예측의 `next_bike_count_pred`입니다.
- `next_bike_count_pred`
  - 현재 총 대수에 `bike_change_pred`를 반영해 계산한 다음 시간 총 대수 예측값입니다.
  - 재귀형 루프의 핵심 상태값입니다.
- `forecast`
  - 시간 순서대로 예측 결과가 쌓이는 배열입니다.
  - 최대 168개 step까지 생성할 수 있습니다.

In [14]:
def recursive_forecast(station_id, start_dt, initial_bike_count, horizon_hours=168):
    horizon_hours = int(min(max(horizon_hours, 1), 24 * 7))
    bundle = MODEL_REGISTRY[int(station_id)]
    records = []
    bike_count = float(initial_bike_count)

    for step in range(1, horizon_hours + 1):
        forecast_dt = start_dt + pd.Timedelta(hours=step)
        pred = predict_station_flow(station_id, forecast_dt)
        next_bike_count = max(bike_count + pred['prediction']['bike_change_pred'], 0.0)
        records.append({
            'step': step,
            'datetime': forecast_dt.isoformat(),
            'bike_count_from_api_or_prev_step': float(bike_count),
            'rental_count_pred': float(pred['prediction']['rental_count_pred']),
            'return_count_pred': float(pred['prediction']['return_count_pred']),
            'bike_change_pred': float(pred['prediction']['bike_change_pred']),
            'next_bike_count_pred': float(next_bike_count),
            'display': {
                'bike_count_from_api_or_prev_step': int(round(bike_count)),
                'rental_count_pred': int(round(pred['prediction']['rental_count_pred'])),
                'return_count_pred': int(round(pred['prediction']['return_count_pred'])),
                'bike_change_pred': int(round(pred['prediction']['bike_change_pred'])),
                'next_bike_count_pred': int(round(next_bike_count)),
            },
        })
        bike_count = next_bike_count

    return {
        'station_id': int(station_id),
        'station_name': bundle['station_name'],
        'start_datetime': start_dt.isoformat(),
        'initial_bike_count_from_api': float(initial_bike_count),
        'horizon_hours': horizon_hours,
        'forecast': records,
    }


## 9. 재귀형 예측 JSON 출력 예시

예시로 현재 API 수량을 `12대`라고 가정하고, 24시간 예측 결과를 JSON으로 출력합니다.

실서비스에서는 `horizon_hours=168`로 주면 최대 1주일 예측이 가능합니다.

In [15]:
start_dt = pd.Timestamp('2026-03-20 10:00:00')
weekly_result = recursive_forecast(TOP6_STATION_IDS[0], start_dt, initial_bike_count=12.0, horizon_hours=24)
print(json.dumps(weekly_result, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "station_name": " 포스코사거리(기업은행)",
  "start_datetime": "2026-03-20T10:00:00",
  "initial_bike_count_from_api": 12.0,
  "horizon_hours": 24,
  "forecast": [
    {
      "step": 1,
      "datetime": "2026-03-20T11:00:00",
      "bike_count_from_api_or_prev_step": 12.0,
      "rental_count_pred": 2.3588602046832117,
      "return_count_pred": 2.1128631409119727,
      "bike_change_pred": 0.24599706377123898,
      "next_bike_count_pred": 12.245997063771238,
      "display": {
        "bike_count_from_api_or_prev_step": 12,
        "rental_count_pred": 2,
        "return_count_pred": 2,
        "bike_change_pred": 0,
        "next_bike_count_pred": 12
      }
    },
    {
      "step": 2,
      "datetime": "2026-03-20T12:00:00",
      "bike_count_from_api_or_prev_step": 12.245997063771238,
      "rental_count_pred": 2.000399666371788,
      "return_count_pred": 2.603551561457781,
      "bike_change_pred": -0.6031518950859933,
      "next_bike_count_pred": 11.6428451

## 10. JSON 결과를 서비스에 적용하는 방법

JSON 결과가 나오면 실제 서비스에서는 이 값을 바로 화면 표시용으로 쓰거나, 다음 step 계산 입력값으로 넘깁니다.

### 단일 시점 예측 결과 적용

- `rental_count_pred`: 해당 시간 예상 대여량
- `return_count_pred`: 해당 시간 예상 반납량
- `bike_change_pred = rental - return`

이 값은 대시보드, 알림, 운영 판단에 바로 사용할 수 있습니다.

### 재귀형 예측 결과 적용

재귀형 예측에서는 각 step의 JSON에서 아래 값을 사용합니다.

- `bike_count_from_api_or_prev_step`: 현재 시간 기준 자전거 수량
- `bike_change_pred`: 다음 1시간 변화량
- `next_bike_count_pred`: 다음 시간 예측 자전거 수량

즉 `next_bike_count_pred`를 다음 step의 입력값으로 계속 넘기면서 최대 7일까지 확장합니다.

In [16]:
service_apply_example = {
    'current_dashboard_value': weekly_result['forecast'][0]['bike_count_from_api_or_prev_step'],
    'next_hour_predicted_bike_count': weekly_result['forecast'][0]['next_bike_count_pred'],
    'next_hour_expected_rental': weekly_result['forecast'][0]['rental_count_pred'],
    'next_hour_expected_return': weekly_result['forecast'][0]['return_count_pred'],
    'next_hour_expected_change': weekly_result['forecast'][0]['bike_change_pred'],
}
print(json.dumps(service_apply_example, ensure_ascii=False, indent=2))

{
  "current_dashboard_value": 12.0,
  "next_hour_predicted_bike_count": 12.245997063771238,
  "next_hour_expected_rental": 2.3588602046832117,
  "next_hour_expected_return": 2.1128631409119727,
  "next_hour_expected_change": 0.24599706377123898
}


## 11. 2026 Dynamic Year Weight 보정 개념

현재 `year_weight`는 2023~2025 학습 결과를 기반으로 만들어졌습니다. 그래서 2026 이후 운영에서는 고정된 `year_weight`만으로는 부족할 수 있습니다.

이 경우 운영에서는 아래와 같은 **동적 연도 보정값(dynamic year_weight)** 을 추가로 둡니다.

- 초기값: `1.0` 또는 `2025 year_weight`
- 예측값 생성
- 실제값 수집
- `actual / predicted` 비율을 계산
- 그 비율을 바로 다 반영하지 않고 이동평균 방식으로 조금씩 업데이트
- 업데이트된 보정값을 다음 예측부터 반영

즉 구조는 아래처럼 됩니다.

`최종예측 = 기존예측 × dynamic_year_weight_2026`

이 방식이면 2026년 운영 데이터가 들어올수록 보정값이 실제 패턴에 적응합니다.

### Dynamic year weight 항목 설명

- `dynamic_year_weight['rental_count']`
  - 2026년 이후 운영에서 대여 예측을 추가로 보정하는 계수입니다.
  - 기본값은 `1.0`이며, 실제 운영 데이터에 따라 계속 갱신됩니다.
- `dynamic_year_weight['return_count']`
  - 2026년 이후 운영에서 반납 예측을 추가로 보정하는 계수입니다.
  - 대여와 반납은 패턴이 다를 수 있으므로 별도로 관리합니다.
- `alpha`
  - 새로 관측된 실제값을 얼마나 빠르게 반영할지 결정하는 학습률입니다.
  - 값이 크면 빠르게 적응하고, 값이 작으면 더 안정적으로 움직입니다.
- `observed_ratio`
  - `actual / predicted`로 계산된 관측 비율입니다.
  - 예측보다 실제가 크면 1보다 크고, 실제가 작으면 1보다 작습니다.
- `updated_dynamic_year_weight`
  - `alpha`를 반영한 최종 보정 계수입니다.
  - 다음 예측부터 이 값이 반영됩니다.

In [17]:
dynamic_year_weight_state = {
    station_id: {
        'rental_count': 1.0,
        'return_count': 1.0,
    }
    for station_id in TOP6_STATION_IDS
}
dynamic_year_weight_state

{2348: {'rental_count': 1.0, 'return_count': 1.0},
 2335: {'rental_count': 1.0, 'return_count': 1.0},
 2377: {'rental_count': 1.0, 'return_count': 1.0},
 2384: {'rental_count': 1.0, 'return_count': 1.0},
 2306: {'rental_count': 1.0, 'return_count': 1.0},
 2375: {'rental_count': 1.0, 'return_count': 1.0}}

## 12. 보정 없는 기본 예측과 dynamic year_weight 반영 예측 비교

아래 함수는 기존 예측값에 `dynamic_year_weight_2026`를 곱해서 운영용 예측값을 만듭니다.

In [18]:
def predict_station_flow_with_dynamic_year_weight(station_id, dt, dynamic_state):
    base_result = predict_station_flow(station_id, dt)
    state = dynamic_state[int(station_id)]

    rental_base = float(base_result['prediction']['rental_count_pred'])
    return_base = float(base_result['prediction']['return_count_pred'])

    rental_adj = max(rental_base * float(state['rental_count']), 0.0)
    return_adj = max(return_base * float(state['return_count']), 0.0)
    base_change = float(rental_base - return_base)
    adjusted_change = float(rental_adj - return_adj)

    return {
        'station_id': int(station_id),
        'station_name': base_result['station_name'],
        'datetime': base_result['datetime'],
        'dynamic_year_weight': {
            'rental_count': float(state['rental_count']),
            'return_count': float(state['return_count']),
        },
        'base_prediction': {
            'rental_count_pred': rental_base,
            'return_count_pred': return_base,
            'bike_change_pred': base_change,
            'display': {
                'rental_count_pred': int(round(rental_base)),
                'return_count_pred': int(round(return_base)),
                'bike_change_pred': int(round(base_change)),
            },
        },
        'adjusted_prediction': {
            'rental_count_pred': float(rental_adj),
            'return_count_pred': float(return_adj),
            'bike_change_pred': adjusted_change,
            'display': {
                'rental_count_pred': int(round(rental_adj)),
                'return_count_pred': int(round(return_adj)),
                'bike_change_pred': int(round(adjusted_change)),
            },
        },
    }


In [19]:
dynamic_example_dt = pd.Timestamp('2026-03-20 11:00:00')
dynamic_example_result = predict_station_flow_with_dynamic_year_weight(TOP6_STATION_IDS[0], dynamic_example_dt, dynamic_year_weight_state)
print(json.dumps(dynamic_example_result, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "station_name": " 포스코사거리(기업은행)",
  "datetime": "2026-03-20T11:00:00",
  "dynamic_year_weight": {
    "rental_count": 1.0,
    "return_count": 1.0
  },
  "base_prediction": {
    "rental_count_pred": 2.3588602046832117,
    "return_count_pred": 2.1128631409119727,
    "bike_change_pred": 0.24599706377123898,
    "display": {
      "rental_count_pred": 2,
      "return_count_pred": 2,
      "bike_change_pred": 0
    }
  },
  "adjusted_prediction": {
    "rental_count_pred": 2.3588602046832117,
    "return_count_pred": 2.1128631409119727,
    "bike_change_pred": 0.24599706377123898,
    "display": {
      "rental_count_pred": 2,
      "return_count_pred": 2,
      "bike_change_pred": 0
    }
  }
}


## 13. 실제값이 들어오면 dynamic year_weight 업데이트

실제 관측값이 들어오면 `actual / predicted` 비율을 계산하고, 그 값을 이동평균 방식으로 반영합니다.

예를 들어 `alpha=0.2`이면 새 관측의 20%만 반영하고, 기존 상태를 80% 유지합니다.

업데이트 식:

`new_weight = old_weight * (1 - alpha) + observed_ratio * alpha`

In [20]:
def update_dynamic_year_weight(station_id, predicted_result, actual_rental, actual_return, dynamic_state, alpha=0.2):
    station_id = int(station_id)
    state = dynamic_state[station_id]

    pred_rental = max(float(predicted_result['adjusted_prediction']['rental_count_pred']), 1e-6)
    pred_return = max(float(predicted_result['adjusted_prediction']['return_count_pred']), 1e-6)

    rental_ratio = float(actual_rental) / pred_rental
    return_ratio = float(actual_return) / pred_return

    old_rental_weight = float(state['rental_count'])
    old_return_weight = float(state['return_count'])

    new_rental_weight = old_rental_weight * (1 - alpha) + rental_ratio * alpha
    new_return_weight = old_return_weight * (1 - alpha) + return_ratio * alpha

    dynamic_state[station_id]['rental_count'] = float(new_rental_weight)
    dynamic_state[station_id]['return_count'] = float(new_return_weight)

    return {
        'station_id': station_id,
        'alpha': float(alpha),
        'observed_ratio': {
            'rental_count': float(rental_ratio),
            'return_count': float(return_ratio),
        },
        'updated_dynamic_year_weight': {
            'rental_count': float(new_rental_weight),
            'return_count': float(new_return_weight),
        },
    }


## 14. 보정 업데이트 예시

예시로 2026-03-20 11시 예측 후 실제 관측값이 아래처럼 들어왔다고 가정합니다.

- 실제 `rental_count = 3.1`
- 실제 `return_count = 1.8`

이 실제값으로 `dynamic_year_weight_2026`를 갱신합니다.

In [21]:
observed_actual = {
    'rental_count': 3.1,
    'return_count': 1.8,
}

update_result = update_dynamic_year_weight(
    station_id=TOP6_STATION_IDS[0],
    predicted_result=dynamic_example_result,
    actual_rental=observed_actual['rental_count'],
    actual_return=observed_actual['return_count'],
    dynamic_state=dynamic_year_weight_state,
    alpha=0.2,
)
print(json.dumps(update_result, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "alpha": 0.2,
  "observed_ratio": {
    "rental_count": 1.314194030593823,
    "return_count": 0.8519245592135551
  },
  "updated_dynamic_year_weight": {
    "rental_count": 1.0628388061187646,
    "return_count": 0.970384911842711
  }
}


## 15. 보정 후 다음 시간 예측

이제 방금 업데이트한 `dynamic_year_weight_2026`를 반영해서 다음 시간 예측을 다시 수행합니다.

즉 흐름은 아래와 같습니다.

1. 기존 예측 수행
2. 실제값 수집
3. dynamic year weight 업데이트
4. 다음 시간 예측에 새 weight 반영

In [22]:
next_dt = pd.Timestamp('2026-03-20 12:00:00')
next_prediction_after_update = predict_station_flow_with_dynamic_year_weight(
    TOP6_STATION_IDS[0],
    next_dt,
    dynamic_year_weight_state,
)
print(json.dumps(next_prediction_after_update, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "station_name": " 포스코사거리(기업은행)",
  "datetime": "2026-03-20T12:00:00",
  "dynamic_year_weight": {
    "rental_count": 1.0628388061187646,
    "return_count": 0.970384911842711
  },
  "base_prediction": {
    "rental_count_pred": 2.000399666371788,
    "return_count_pred": 2.603551561457781,
    "bike_change_pred": -0.6031518950859933,
    "display": {
      "rental_count_pred": 2,
      "return_count_pred": 3,
      "bike_change_pred": -1
    }
  },
  "adjusted_prediction": {
    "rental_count_pred": 2.126102393166966,
    "return_count_pred": 2.5264471524431618,
    "bike_change_pred": -0.4003447592761957,
    "display": {
      "rental_count_pred": 2,
      "return_count_pred": 3,
      "bike_change_pred": 0
    }
  }
}


## 15-1. 동적 연도 보정 전체 결과를 하나의 JSON으로 출력

아래 JSON은 `보정 전 예측`, `실제값 기반 업데이트`, `보정 후 다음 시간 예측`을 한 번에 묶은 형태입니다.

In [23]:
dynamic_year_weight_full_result = {
    'station_id': int(TOP6_STATION_IDS[0]),
    'station_name': MODEL_REGISTRY[int(TOP6_STATION_IDS[0])]['station_name'],
    'before_update_prediction': dynamic_example_result,
    'update_result': update_result,
    'after_update_next_hour_prediction': next_prediction_after_update,
}
print(json.dumps(dynamic_year_weight_full_result, ensure_ascii=False, indent=2))

{
  "station_id": 2348,
  "station_name": " 포스코사거리(기업은행)",
  "before_update_prediction": {
    "station_id": 2348,
    "station_name": " 포스코사거리(기업은행)",
    "datetime": "2026-03-20T11:00:00",
    "dynamic_year_weight": {
      "rental_count": 1.0,
      "return_count": 1.0
    },
    "base_prediction": {
      "rental_count_pred": 2.3588602046832117,
      "return_count_pred": 2.1128631409119727,
      "bike_change_pred": 0.24599706377123898,
      "display": {
        "rental_count_pred": 2,
        "return_count_pred": 2,
        "bike_change_pred": 0
      }
    },
    "adjusted_prediction": {
      "rental_count_pred": 2.3588602046832117,
      "return_count_pred": 2.1128631409119727,
      "bike_change_pred": 0.24599706377123898,
      "display": {
        "rental_count_pred": 2,
        "return_count_pred": 2,
        "bike_change_pred": 0
      }
    }
  },
  "update_result": {
    "station_id": 2348,
    "alpha": 0.2,
    "observed_ratio": {
      "rental_count": 1.3141940305938

## 16. 운영 흐름 전체 요약

2026년 이후 운영에서는 아래 루프를 반복하면 됩니다.

1. 현재 시각 기준 예측 수행
2. 예측 결과를 JSON으로 반환
3. 다음 시간 실제값이 들어오면 dynamic year weight 업데이트
4. 업데이트된 weight로 다음 예측 수행
5. 이 과정을 계속 반복해서 2026년 이후 패턴 변화에 적응

즉, 고정된 2023~2025 `year_weight` 위에 **운영 중 적응하는 2026 보정 계층**을 하나 더 얹는 방식입니다.

### 실서비스 절차를 항목 기준으로 다시 정리

1. 클라이언트 또는 스케줄러가 `station_id`, `datetime`, `current_bike_count_from_api`를 보냅니다.
2. 서버는 `station_id`로 해당 station bundle을 메모리에서 찾습니다.
3. `datetime`을 사용해 내부 feature를 계산합니다.
4. `rental_count_pred`, `return_count_pred`를 float 값으로 예측합니다.
5. `bike_change_pred`와 `next_bike_count_pred`를 계산합니다.
6. 응답 JSON에는 `prediction`과 `display`를 함께 넣어 반환합니다.
7. 실제 관측값이 들어오면 `dynamic_year_weight`를 업데이트합니다.
8. 다음 요청부터는 업데이트된 `dynamic_year_weight`를 반영합니다.